In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture # <--- NEW IMPORT
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

df_lt

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8985.274630,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,10219.116755,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,8528.034573,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,8931.519313,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10259.331961,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11120.403233,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11776.792571,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12395.979849,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12861.323296,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13491.939909,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [5]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['CO2C_3'] = lt['CO2C'].shift(3)
lt['VABI_2'] = lt['VABI'].shift(2)
lt['QOA_3'] = lt['QOA'].shift(3)
lt['EBSG_3'] = lt['EBSG'].shift(3)
lt['FDI_1'] = lt['FDI'].shift(1)
lt['ARP_3'] = lt['ARP'].shift(3)

In [6]:
lt = lt.dropna()

In [7]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='ltnse').astype(int) - 1
    num_states = discretised_lt[col].nunique()
    
    if num_states == 2:
        discretised_lt[col] = discretised_lt[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lt[col] = discretised_lt[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lt[col] = discretised_lt[col].astype(str)

print("Discretised data:")
print(discretised_lt)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD  ...   ARP  \
3    Low  High   Low   Low  High  High   Low   Low  High   Low  ...   Low   
4    Low  High   Low   Low  High  High   Low   Low  High   Low  ...   Low   
5    Low  High   Low   Low  High  High   Low   Low  High   Low  ...   Low   
6    Low  High   Low  High  High  High   Low   Low   Low   Low  ...   Low   
7    Low  High   Low   Low   Low  High   Low  High   Low   Low  ...   Low   
8    Low  High   Low   Low   Low   Low  High   Low  High  High  ...  High   
9    Low   Low   Low  High   Low   Low  High   Low   Low  High  ...  High   
10  High   Low  High  High   Low   Low  High  High   Low  High  ...  High   
11  High   Low  High  High   Low   Low   Low  High   Low  High  ...  High   
12  High   Low  High  High   Low   Low   Low  High   Low  High  ...   Low   
13  High   Low  High   Low   Low  High  High  High  High   Low  ...  High   
14  High   Low  High   Low   Low   Low  High  High  High  

In [8]:
sm = StructureModel()

In [9]:
sm.add_edges_from([  
   ('EUPC', 'PC'),
    ('CO2C_3', 'PC'),
    ('VABI_2', 'EUPC'),
    ('QOA_3', 'EUPC'),
    ('EBSG_3', 'EUPC'),
    ('FDI_1', 'EUPC'),
    ('ARP_3', 'EUPC')
])



In [10]:
sm.edges

OutEdgeView([('EUPC', 'PC'), ('CO2C_3', 'PC'), ('VABI_2', 'EUPC'), ('QOA_3', 'EUPC'), ('EBSG_3', 'EUPC'), ('FDI_1', 'EUPC'), ('ARP_3', 'EUPC')])

In [11]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [12]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lt)

bn.fit_cpds(discretised_lt, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lt)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 4484.86 (LogLikelihood: -2155.22)


In [13]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'PC'],
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'PC']
  - Removing direct edge: EUPC -> PC
LV Model BIC: 5187.82 (LogLikelihood: -2424.62)


In [14]:
bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['VABI_2', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['VABI_2', 'EUPC']
  - Removing direct edge: VABI_2 -> EUPC
LV Model BIC: 5028.79 (LogLikelihood: -2424.62)


In [15]:
bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['QOA_3', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['QOA_3', 'EUPC']
  - Removing direct edge: QOA_3 -> EUPC
LV Model BIC: 5028.79 (LogLikelihood: -2424.62)


In [16]:
bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EBSG_3', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['EBSG_3', 'EUPC']
  - Removing direct edge: EBSG_3 -> EUPC
LV Model BIC: 5028.79 (LogLikelihood: -2424.62)


In [17]:
bn_with_lv_5, data_with_lv_5, bic_lv_5 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['FDI_1', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['FDI_1', 'EUPC']
  - Removing direct edge: FDI_1 -> EUPC
LV Model BIC: 5028.79 (LogLikelihood: -2424.62)


In [18]:
bn_with_lv_6, data_with_lv_6, bic_lv_6 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['ARP_3', 'EUPC'],
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['ARP_3', 'EUPC']
  - Removing direct edge: ARP_3 -> EUPC
LV Model BIC: 5028.79 (LogLikelihood: -2424.62)


In [29]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [30]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

df_lt

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8985.274630,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,10219.116755,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,8528.034573,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,8931.519313,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10259.331961,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11120.403233,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11776.792571,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12395.979849,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12861.323296,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13491.939909,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [32]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['EUPC_2'] = lt['EUPC'].shift(2)
lt['ARP_2'] = lt['ARP'].shift(2)
lt['FDI_3'] = lt['FDI'].shift(3)
lt['VABI_1'] = lt['VABI'].shift(1)
lt['REST_1'] = lt['REST'].shift(1)
lt['PAM_3'] = lt['PAM'].shift(3)
lt['EBSG_2'] = lt['EBSG'].shift(2)

In [33]:
lt = lt.dropna()

In [34]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='ltnse').astype(int) - 1
    num_states = discretised_lt[col].nunique()
    
    if num_states == 2:
        discretised_lt[col] = discretised_lt[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lt[col] = discretised_lt[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lt[col] = discretised_lt[col].astype(str)

print("Discretised data:")
print(discretised_lt)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD  ...  CO2C  \
3    Low  High   Low   Low  High  High   Low   Low  High   Low  ...   Low   
4    Low  High   Low   Low  High  High   Low   Low  High   Low  ...   Low   
5    Low  High   Low   Low  High  High   Low   Low  High   Low  ...  High   
6    Low  High   Low  High  High  High   Low   Low   Low   Low  ...   Low   
7    Low  High   Low   Low   Low  High   Low  High   Low   Low  ...   Low   
8    Low  High   Low   Low   Low   Low  High   Low  High  High  ...   Low   
9    Low   Low   Low  High   Low   Low  High   Low   Low  High  ...   Low   
10  High   Low  High  High   Low   Low  High  High   Low  High  ...  High   
11  High   Low  High  High   Low   Low   Low  High   Low  High  ...  High   
12  High   Low  High  High   Low   Low   Low  High   Low  High  ...  High   
13  High   Low  High   Low   Low  High  High  High  High   Low  ...  High   
14  High   Low  High   Low   Low   Low  High  High  High  

In [35]:
sm = StructureModel()

In [36]:
sm.add_edges_from([
     ('EUPC_2', 'FDI'),
    ('ARP_2', 'FDI'),
    ('ARP_2', 'CO2C'),
    ('FDI_3', 'CO2C'),
    ('VABI_1', 'FDI'),
    ('VABI_1', 'ARP'),
    ('FDI_3', 'PAM'),
    ('REST_1', 'PAM'),
    ('REST_1', 'QOR'),
    ('PAM_3', 'ARP'),
     ('EBSG_2', 'PAM'),
    ('FDI_3', 'PAM')
])

In [37]:
sm.edges

OutEdgeView([('EUPC_2', 'FDI'), ('ARP_2', 'FDI'), ('ARP_2', 'CO2C'), ('FDI_3', 'CO2C'), ('FDI_3', 'PAM'), ('VABI_1', 'FDI'), ('VABI_1', 'ARP'), ('REST_1', 'PAM'), ('REST_1', 'QOR'), ('PAM_3', 'ARP'), ('EBSG_2', 'PAM')])

In [38]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [39]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lt)

bn.fit_cpds(discretised_lt, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lt)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 6573.39 (LogLikelihood: -3232.83)


In [40]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'FDI'],  # Replace with your specific nodes
    discretised_data=discretised_lt,
    raw_data=lt
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'FDI']
  - Removing direct edge: EUPC_2 -> FDI
LV Model BIC: 7117.32 (LogLikelihood: -3502.23)
